# 03 - Circuit Partitioning with QONTOS

When a quantum circuit exceeds the capacity of a single quantum module,
QONTOS **partitions** it into smaller sub-circuits that can be executed
independently on separate hardware modules. The results are then stitched
back together by the `ResultAggregator`.

This notebook demonstrates:

1. Building a **10-qubit** circuit with non-trivial connectivity
2. Partitioning with the **Greedy** strategy
3. Inspecting the partition plan and qubit-to-partition mapping
4. Comparing **Spectral** vs **Greedy** tradeoffs

## Environment

This notebook works with the QONTOS local simulator. No quantum hardware
access is needed.

```bash
# pip install git+https://github.com/qontos/qontos.git@v0.2.0 git+https://github.com/qontos/qontos-sim.git@v0.1.0
```

## Prerequisites

- Python >= 3.10
- `qontos` SDK installed
- Familiarity with [01_hello_qubit](01_hello_qubit.ipynb) and [02_bell_state](02_bell_state.ipynb)

In [ ]:
from qontos.circuit import CircuitNormalizer
from qontos.partitioning import Partitioner, PartitionConstraints
from qontos.partitioning.models import PartitionStrategy

## Build a 10-Qubit Circuit

We construct a circuit with a mix of nearest-neighbour and long-range CNOT
gates. This topology makes partitioning non-trivial because the partitioner
must decide where to cut the circuit graph to minimize inter-module
communication.

In [ ]:
NUM_QUBITS = 10

lines = [
    "OPENQASM 2.0;",
    'include "qelib1.inc";',
    f"qreg q[{NUM_QUBITS}];",
    f"creg c[{NUM_QUBITS}];",
]

# Layer 1: Hadamard on all qubits
for i in range(NUM_QUBITS):
    lines.append(f"h q[{i}];")

# Layer 2: Nearest-neighbour CNOT chain
for i in range(NUM_QUBITS - 1):
    lines.append(f"cx q[{i}],q[{i + 1}];")

# Layer 3: Long-range connections (skip-2)
for i in range(0, NUM_QUBITS - 2, 2):
    lines.append(f"cx q[{i}],q[{i + 2}];")

# Layer 4: Rotation layer
for i in range(NUM_QUBITS):
    lines.append(f"rz(0.3) q[{i}];")

# Layer 5: Reverse chain
for i in range(NUM_QUBITS - 1, 0, -1):
    lines.append(f"cx q[{i}],q[{i - 1}];")

# Measurements
for i in range(NUM_QUBITS):
    lines.append(f"measure q[{i}] -> c[{i}];")

qasm_10q = "\n".join(lines) + "\n"
print(f"Circuit has {NUM_QUBITS} qubits")
print(f"QASM lines: {len(lines)}")

In [ ]:
# Normalize to CircuitIR
normalizer = CircuitNormalizer()
circuit_ir = normalizer.normalize(input_type="openqasm", source=qasm_10q)

print(f"Qubits          : {circuit_ir.num_qubits}")
print(f"Gate count       : {circuit_ir.gate_count}")
print(f"Circuit depth    : {circuit_ir.depth}")
print(f"Connectivity     : {len(circuit_ir.qubit_connectivity)} edges")

## Partition with Greedy Strategy

The **Greedy** partitioner grows partitions outward from high-degree seed
qubits. It is fast and produces good results for small-to-medium circuits.

We request **3 partitions** to simulate distributing across three quantum modules.

In [ ]:
constraints = PartitionConstraints(
    target_partitions=3,
    preferred_strategy=PartitionStrategy.GREEDY,
)

partitioner = Partitioner()
greedy_plan = partitioner.run(
    circuit_ir,
    job_id="notebook-greedy-demo",
    constraints=constraints,
)

print(f"Strategy           : {greedy_plan.strategy}")
print(f"Num partitions     : {greedy_plan.estimated_module_count}")
print(f"Inter-module gates : {greedy_plan.total_inter_module_gates}")
print(f"Balance score      : {greedy_plan.partition_balance_score:.4f}")
print(f"Cut ratio          : {greedy_plan.cut_ratio:.4f}")

## Inspect the Partition Plan

Each partition contains a subset of qubits and gates. The `qubit_mapping`
translates global qubit indices to local indices within the partition.

In [ ]:
for entry in greedy_plan.partitions:
    print(f"Partition {entry.partition_index}:")
    print(f"  ID              : {entry.partition_id}")
    print(f"  Qubits (global) : {entry.qubit_indices}")
    print(f"  Num qubits      : {entry.num_qubits}")
    print(f"  Gate count      : {entry.gate_count}")
    print(f"  Qubit mapping   : {entry.qubit_mapping}")
    print()

## Visualize Qubit-to-Partition Mapping

A simple visualization showing which qubits belong to which partition.
In a real deployment, each partition maps to a physical quantum module.

In [ ]:
# Build qubit -> partition mapping
qubit_to_partition = {}
for entry in greedy_plan.partitions:
    for q in entry.qubit_indices:
        qubit_to_partition[q] = entry.partition_index

# Display as a visual bar
colors = ["A", "B", "C", "D", "E"]  # partition labels
print("Qubit-to-Partition Mapping:")
print()
for q in range(NUM_QUBITS):
    p = qubit_to_partition.get(q, "?")
    label = colors[p] if isinstance(p, int) and p < len(colors) else "?"
    bar = "##" * 4
    print(f"  q{q:>2} -> Partition {p} [{label}] {bar}")

print()
print("Legend:")
for entry in greedy_plan.partitions:
    label = colors[entry.partition_index]
    print(f"  [{label}] Partition {entry.partition_index}: qubits {entry.qubit_indices}")

## Spectral vs Greedy: Tradeoffs

QONTOS supports two partitioning strategies:

| Property | Greedy | Spectral |
|----------|--------|----------|
| Algorithm | Seed expansion from high-degree qubits | Fiedler vector of graph Laplacian |
| Speed | Fast (linear in gate count) | Slower (eigenvalue computation) |
| Best for | Small circuits (< 20 qubits) | Large circuits (>= 20 qubits) |
| Optimality | May get stuck in local optima | Globally-informed bisection |
| Balance | Good but not guaranteed | Tends to produce well-balanced cuts |

Let's compare both strategies on our 10-qubit circuit.

In [ ]:
# Partition with Spectral strategy
spectral_constraints = PartitionConstraints(
    target_partitions=3,
    preferred_strategy=PartitionStrategy.SPECTRAL,
)
spectral_plan = partitioner.run(
    circuit_ir,
    job_id="notebook-spectral-demo",
    constraints=spectral_constraints,
)

# Side-by-side comparison
print(f"{'Metric':<28} {'Greedy':>10} {'Spectral':>10}")
print("-" * 50)
print(f"{'Inter-module gates':<28} {greedy_plan.total_inter_module_gates:>10} "
      f"{spectral_plan.total_inter_module_gates:>10}")
print(f"{'Balance score':<28} {greedy_plan.partition_balance_score:>10.4f} "
      f"{spectral_plan.partition_balance_score:>10.4f}")
print(f"{'Cut ratio':<28} {greedy_plan.cut_ratio:>10.4f} "
      f"{spectral_plan.cut_ratio:>10.4f}")
print(f"{'Comm overhead (us)':<28} {greedy_plan.estimated_communication_overhead_us:>10.1f} "
      f"{spectral_plan.estimated_communication_overhead_us:>10.1f}")

## Summary

In this notebook you learned:

- How to build larger circuits programmatically in OpenQASM
- How `Partitioner.run()` splits a circuit into modules
- How to inspect partition plans: qubit assignments, gate counts, mappings
- The tradeoffs between **Greedy** (fast, local) and **Spectral** (slower, global)

**Next:** [04_multi_backend.ipynb](04_multi_backend.ipynb) shows how QONTOS
schedules partitions across different quantum backends.